# Executive Summary

**Objective:** 
To integrate geographical metadata, resolve string formatting inconsistencies, and engineer normalized metrics, transforming the raw data into a finalized state ready for exploratory data analysis.

**Data Flow:**
*   **Inputs:** 
    * `data\raw\internship_positions.parquet`
    * `data\external\administrative_divisions.parquet`
*   **Output:** `data\interim\internship_postings.parquet` (Exported to the interim directory to preserve datatypes)

**Key Operations Performed:**
1. **Data Integration (Province Mapping) [<u>[click]</u>](#1-data-integration):** Mapped raw job locations to their respective provinces by joining against the administrative divisions dictionary.
    * **String Cleaning [<u>[click]</u>](#11-string-cleaning):** Resolved mismatching geographic keys by standardizing text (lowercasing, stripping punctuation, and removing extraneous words).
    * **Manual Overrides [<u>[click]</u>](#12-manual-overrides):** Applied a manual dictionary mapping to achieve 100% province mapping with zero null values.
2. **Data Conversion [<u>[click]</u>](#2-data-conversion):** Cast `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).
3. **Feature Engineering [<u>[click]</u>](#3-feature-engineering):**
    * Engineered the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`).
    * Binned all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.
    * One-hot encoded `education_level` for downstream stakeholder consumption.
    * Extracted a new binary flag, `allows_all_majors`, by parsing the `job_description` column.
4. **Schema Finalization [<u>[click]</u>](#4-schema-finalization):** Reorganized the final 14 columns into a logical analytical structure before exporting.

# Setup & Imports

In [87]:
# Import libraries
import numpy as np
import pandas as pd

from src.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR, RAW_DATA_DIR

In [88]:
# Load datasets
adm_divisions = pd.read_parquet(EXTERNAL_DATA_DIR / "administrative_divisions.parquet")
internship_positions = pd.read_parquet(RAW_DATA_DIR / "internship_positions.parquet")

# 1. Data Integration
Mapping raw job locations to their respective provinces by joining against the administrative divisions dictionary.

In [89]:
# Convert a regency-province table into a dictionary
adm_divisions_dict = dict(zip(adm_divisions["regency"], adm_divisions["province"]))


In [90]:
# Create a new column, `province`, by mapping with a dictionary
internship_positions["province"] = internship_positions["job_location"].map(adm_divisions_dict)
display(internship_positions.head())

,job_id,published_at,job_title,company,job_location,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur


In [91]:
# Find regencies from `internship_positions` that have no match 
# with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Kota Batam                          221
Kota Palangkaraya                   120
Kota Dumai                           77
Kab. Pangkajene Kepulauan            57
Kota Bau Bau                         56
Kota Banjarbaru                      48
Kab. Siak                            46
Kota Lubuk Linggau                   43
Kab. Gunungkidul                     42
Kab. Karangasem                      39
Kota Sawahlunto                      38
Kab. Batanghari                      31
Kab. Tulang Bawang                   28
Kota Pematangsiantar                 25
Unknown Location                     24
Kab. Kotabaru                        24
Kab. Banyuasin                       23
Kab. Labuhanbatu                     23
Kota Pare Pare                       20
Kab. Tojo Una Una                    18
Kab. Pahuwato                        17
Kab. Kep. Siau Tagulandang Biaro     17
Kepulauan Tanimbar                   15
Kab Timor Tengah Selatan             15
Kab. Toli Toli             

In [92]:
# Get all regencies from `adm_divisions` that have no match
# with job locations from `internship_positions`
matched_regencies = list(set(internship_positions[
    ~internship_positions.province.isnull()
]["job_location"].to_list()))

unmatched_regencies = list(set(adm_divisions.regency.to_list()) - set(matched_regencies))

display(
    adm_divisions[
        adm_divisions.regency.isin(unmatched_regencies)
    ][["regency", "province"]].sort_values("regency", ascending=False)
)

,regency,province
70,Kota Sawah Lunto,Sumatera Barat
50,Kota Pematang Siantar,Sumatera Utara
420,Kota Parepare,Sulawesi Selatan
341,Kota Palangka Raya,Kalimantan Tengah
114,Kota Lubuklinggau,Sumatera Selatan
...,...,...
90,Kab. Batang Hari,Jambi
104,Kab. Banyu Asin,Sumatera Selatan
144,Kab. Bangka Selatan,Kepulauan Bangka Belitung
385,Kab. Banggai Kepulauan,Sulawesi Tengah


## 1.1 String Cleaning

In [93]:
# Fallback mapping: Stripping punctuation and whitespace resolves mismatches
# caused by inconsistent data entry on the scraped platform.
plain_adm_div_dict = dict(
    zip(
        adm_divisions["regency"]
        .str.lower()
        .str.replace(r"\sdan\s", " ", case=False, regex=True)
        .str.replace(r"\W", "", regex=True),
        adm_divisions["province"],
    )
)

display(plain_adm_div_dict)

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = (
    internship_positions.loc[mask, "job_location"]
    .str.lower()
    .str.replace(r"\sdan\s", " ", case=False, regex=True)
    .str.replace(r"\W", "", regex=True)
    .map(plain_adm_div_dict)
)

{'kabsimeulue': 'Aceh',
 'kabacehsingkil': 'Aceh',
 'kabacehselatan': 'Aceh',
 'kabacehtenggara': 'Aceh',
 'kabacehtimur': 'Aceh',
 'kabacehtengah': 'Aceh',
 'kabacehbarat': 'Aceh',
 'kabacehbesar': 'Aceh',
 'kabpidie': 'Aceh',
 'kabbireuen': 'Aceh',
 'kabacehutara': 'Aceh',
 'kabacehbaratdaya': 'Aceh',
 'kabgayolues': 'Aceh',
 'kabacehtamiang': 'Aceh',
 'kabnaganraya': 'Aceh',
 'kabacehjaya': 'Aceh',
 'kabbenermeriah': 'Aceh',
 'kabpidiejaya': 'Aceh',
 'kotabandaaceh': 'Aceh',
 'kotasabang': 'Aceh',
 'kotalangsa': 'Aceh',
 'kotalhokseumawe': 'Aceh',
 'kotasubulussalam': 'Aceh',
 'kabnias': 'Sumatera Utara',
 'kabmandailingnatal': 'Sumatera Utara',
 'kabtapanuliselatan': 'Sumatera Utara',
 'kabtapanulitengah': 'Sumatera Utara',
 'kabtapanuliutara': 'Sumatera Utara',
 'kabtobasamosir': 'Sumatera Utara',
 'kablabuhanbatu': 'Sumatera Utara',
 'kabasahan': 'Sumatera Utara',
 'kabsimalungun': 'Sumatera Utara',
 'kabdairi': 'Sumatera Utara',
 'kabkaro': 'Sumatera Utara',
 'kabdeliserdang': '

In [94]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

job_location
Unknown Location                    24
Kab. Kep. Siau Tagulandang Biaro    17
Kab. Pahuwato                       17
Kepulauan Tanimbar                  15
Kab. Mahakam Ulu                     1
Name: job_id, dtype: int64

## 1.2 Manual Overrides

In [95]:
# Manual overrides for edge cases missing from the standard division dataset
manual_dict = {
    "Kab. Kep. Siau Tagulandang Biaro": "Sulawesi Utara",
    "Kab. Mahakam Ulu": "Kalimantan Timur",
    "Kab. Pahuwato": "Gorontalo",
    "Kepulauan Tanimbar": "Maluku",
    "Unknown Location": "Unknown Location",
}

mask = internship_positions["province"].isnull()
internship_positions.loc[mask, "province"] = internship_positions.loc[
    mask, "job_location"
].map(manual_dict)

In [96]:
# Check for the missing values again by finding regencies from `internship_positions`
# that have no match with their corresponding provinces
display(
    internship_positions[
        internship_positions["province"].isnull()
    ].groupby("job_location")["job_id"].count().sort_values(ascending=False)
)

Series([], Name: job_id, dtype: int64)

In [97]:
# Rename Column `job_location` to `regency_city`
internship_positions.rename(columns={"job_location": "regency_city"}, inplace=True)

# 2. Data Conversion
Casting `weekly_working_day` to a categorical datatype due to its low cardinality (2 unique values).

In [98]:
# Cast the data type of `weekly_working_day` to string
internship_positions = internship_positions.astype({"weekly_working_day": "str"})

internship_positions.info()

<class 'pandas.DataFrame'>
RangeIndex: 28322 entries, 0 to 28321
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   job_id              28322 non-null  str  
 1   published_at        28322 non-null  str  
 2   job_title           28322 non-null  str  
 3   company             28322 non-null  str  
 4   regency_city        28322 non-null  str  
 5   education_level     28322 non-null  str  
 6   allowed_major       28322 non-null  str  
 7   job_description     28322 non-null  str  
 8   weekly_working_day  28322 non-null  str  
 9   requested_quota     28322 non-null  int64
 10  approved_quota      28322 non-null  int64
 11  applicant_count     28322 non-null  int64
 12  province            28322 non-null  str  
dtypes: int64(3), str(10)
memory usage: 20.0 MB


# 3. Feature Engineering

## 3.1 Feature Construction
Constructing the `acceptance_percentage` column (`100 * approved_quota / (1 + applicant_count)`)

In [99]:
# Create Column `acceptance_percentage`
internship_positions["acceptance_percentage"] = round(
    100
    * internship_positions["approved_quota"]
    / (internship_positions["applicant_count"].add(1)),
    2,
)

display(internship_positions.sample(10))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage
17895,a2334b8f-878e-46d8-9772-14911022d46a,2026-07-16T10:25:01+07:00,akutansi keuangan,Karya Bintang Mandiri,Kab. Sidoarjo,Bachelor,"Akuntansi Perpajakan, Akuntansi Keuangan Perus...",akuntansi keuangan untuk perusahaan jasa menca...,6,1,1,9,Jawa Timur,10.00
14961,a242ea96-f44e-426b-964d-4c911e845bb1,2026-07-16T12:34:12+07:00,Asisten Statistisi,BPS Kabupaten Mamuju,Kab. Mamuju,"Bachelor, Diploma","Sains Data, Ilmu Komputer, Sistem informasi, S...","Membantu pengumpulan, pengolahan, verifikasi, ...",5,4,4,30,Sulawesi Barat,12.90
1087,a23f32dc-2c3c-455c-b8f8-3208de60af16,2026-07-16T10:42:20+07:00,Asisten Apoteker (Tenaga Teknis Kefarmasian),Puskesmas Wanayasa 1,Kab. Banjarnegara,"Diploma, Bachelor, Profession",Farmasi,Melakukan pengelolaan logistik sediaan farmasi...,6,1,1,2,Jawa Tengah,33.33
10339,a2429ce0-aed3-4de7-bec8-5aa38371e25b,2026-07-16T10:58:49+07:00,Sales Support Intern,Tvs Scs Indonesia,Kota Adm. Jakarta Selatan,"Diploma, Bachelor","Manajemen Transportasi, Akuntansi, Bahasa dan ...",Membantu tim Sales dalam kegiatan administrasi...,5,1,1,6,Dki Jakarta,14.29
21992,a2411196-52e7-4353-8859-8da38b83ffce,2026-07-16T10:16:17+07:00,IT Support Staff,Intanwijaya Internasional,Kota Semarang,Bachelor,"Teknik Informatika, Sistem Informasi, Teknik K...","1. Membantu pendataan inventaris perangkat IT,...",5,1,1,12,Jawa Tengah,7.69
27309,a23f5c18-af0d-4fdd-9dde-05b9b59aef09,2026-07-16T11:52:25+07:00,Pengelola Ketahanan Pangan,KANTOR WILAYAH DIREKTORAT JENDERAL PEMASYARAKA...,Kota Jambi,Bachelor,"Ilmu Pertanian, Ilmu Peternakan","""1.\tMenyusun rencana dan melaksanakan program...",5,2,2,49,Jambi,4.00
17376,a23fad4c-ebd7-4efa-916e-51eaabf0df1f,2026-07-16T12:22:53+07:00,Pengelola BMN,LEMBAGA PEMASYARAKATAN KELAS IIB SINGKAWANG,Kota Singkawang,Bachelor,Manajemen Aset Publik,"""1. Mengelola aset dan inventaris milik negara...",5,1,1,9,Kalimantan Barat,10.00
27889,a24183f8-79b7-415d-84da-07b2d56c7f18,2026-07-16T10:12:39+07:00,Staf Budidaya,Sinergi Gula Nusantara,Kota Surabaya,Bachelor,"Ilmu Pertanian, Statistika",mendukung pelaksanaan kegiatan budidaya tebu m...,5,1,1,31,Jawa Timur,3.12
5184,a241d9d5-387b-46f4-b996-bb64ffd27904,2026-07-16T12:30:50+07:00,Duta Layanan,KANIM KELAS III TPI KOTAWARINGIN BARAT,Kab. Kotawaringin Barat,Bachelor,"Hubungan Masyarakat, Ilmu Komunikasi, Pariwisata","""1. Memberikan pelayanan langsung kepada masya...",5,2,2,9,Kalimantan Tengah,20.00
29,a228a0dd-f261-4537-8b20-902eb26478c9,2026-07-23T20:46:28+07:00,D4 PENATA ANASTESI,Rumah Sakit Umum Dr. H. Koesnadi Kabupaten Bon...,Kab. Bondowoso,Bachelor,Keperawatan Anestesiologi,"Pendidikan minimal D4 Penata Anastesi , memaha...",6,2,2,2,Jawa Timur,66.67


## 3.2 Feature Transformation
Binning all heavy right-skewed numericals (`requested_quota`, `approved_quota`, `applicant_count`, `acceptance_percentage`) to capture "whale" postings in an "Extreme" category without deleting them.

In [100]:
# Bin `requested_quota` and `approved_quota`
quota_edges = [1, 2, 10, 50, np.inf]
quota_labels = ["1 to 2", "3 to 10", "11 to 50", "50+"]

internship_positions["requested_quota_category"] = pd.cut(
    internship_positions["requested_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

internship_positions["approved_quota_category"] = pd.cut(
    internship_positions["approved_quota"],
    bins=quota_edges,
    labels=quota_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,1 to 2,1 to 2
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,1 to 2,1 to 2
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,1 to 2,1 to 2


In [101]:
# Bin Column `applicant_count`
applicant_edges = [0, 5, 10, 20, 50, np.inf]
applicant_labels = ["0 to 5", "6 to 10", "11 to 20", "21 to 50", "50+"]

internship_positions["applicant_count_category"] = pd.cut(
    internship_positions["applicant_count"],
    bins=applicant_edges,
    labels=applicant_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,1 to 2,1 to 2,0 to 5
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,1 to 2,1 to 2,0 to 5
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,1 to 2,1 to 2,0 to 5


In [102]:
# Bin Column `acceptance_percentage`
acceptance_edges = [0, 10, 25, 50, np.inf]
acceptance_labels = ["0 - 10%", "11 - 25%", "26 - 50%", "50%+"]

internship_positions["acceptance_percentage_category"] = pd.cut(
    internship_positions["acceptance_percentage"],
    bins=acceptance_edges,
    labels=acceptance_labels,
    include_lowest=True
)

display(internship_positions.head())

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,approved_quota,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category
0,a240f2ba-12c0-4958-b416-c3e9c1d4e344,2026-07-16T12:58:55+07:00,PSIKOLOG,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Psikologi,1. Melakukan asesmen psikologis terhadap anak ...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5,50%+
1,a240f336-b6b7-47fe-8096-5b4c2eb2ed1f,2026-07-16T12:58:55+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SIBUHUAN,Kab. Padang Lawas,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Sumatera Utara,100.0,1 to 2,1 to 2,0 to 5,50%+
2,a242e6a4-2c7a-4ce4-8a57-7aff601a5e2c,2026-07-16T12:57:40+07:00,PSIKIATER,RUMAH TAHANAN NEGARA KELAS IIB SALATIGA,Kota Salatiga,Profession,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,1,0,Jawa Tengah,100.0,1 to 2,1 to 2,0 to 5,50%+
3,a24138aa-bcf6-4940-bfaf-dfe5dbf66ca6,2026-07-16T12:50:42+07:00,PERAWAT KESEHATAN,LEMBAGA PEMASYARAKATAN KELAS III SUKAMARA,Kab. Sukamara,Bachelor,Ilmu Gizi,1. Memberikan perawatan kesehatan umum dan tin...,6,1,1,0,Kalimantan Tengah,100.0,1 to 2,1 to 2,0 to 5,50%+
4,a23f7c52-9123-46e9-bb5e-be376b1d77f2,2026-07-16T12:38:52+07:00,Psikiater,LEMBAGA PEMASYARAKATAN KELAS III ARJASA,Kab. Sumenep,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,6,1,1,0,Jawa Timur,100.0,1 to 2,1 to 2,0 to 5,50%+


## 3.3 Feature Encoding
One-hot encoding `education_level` for downstream stakeholder consumption.

In [103]:
# One hot encode `education_level`
ed_level_dummies = internship_positions["education_level"].str.lower().str.get_dummies(sep=", ")
ed_level_dummies = ed_level_dummies.replace({0: "No", 1: "Yes"}).add_prefix("allows_")
ed_level_dummies = ed_level_dummies.add_suffix("_level")

internship_positions = pd.concat([internship_positions, ed_level_dummies], axis=1)
display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,applicant_count,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level
16418,a24117a2-712b-4b46-952a-771e641541e7,2026-07-16T12:46:21+07:00,PENGELOLA KEGIATAN KERJA,RUMAH TAHANAN NEGARA KELAS I DEPOK,Kota Depok,Bachelor,"Teknologi Pertanian, Ilmu Pertanian, Ekonomi P...",$28,6,2,...,16,Jawa Barat,11.76,1 to 2,1 to 2,11 to 20,11 - 25%,Yes,No,No
14198,a2415bcd-b193-4304-a486-bf702d2b2d9f,2026-07-16T12:44:52+07:00,Koder Rawat Jalan (Casemix),Rumah Sakit Umum Pusat Surabaya,Kota Surabaya,Diploma,Perekam Medis dan Informasi Kesehatan,"""- Mengkoding klaim Rawat jalan \n - Memverifi...",6,2,...,14,Jawa Timur,13.33,1 to 2,1 to 2,11 to 20,11 - 25%,No,Yes,No
5663,a2295a68-b1ab-4b83-bf2f-7fee30658d80,2026-07-16T20:15:41+07:00,IT Developer,Mitra Bakti Ut,Kota Adm. Jakarta Timur,"Bachelor, Diploma","Teknik Informatika, Ilmu Komputer, Sistem Info...",Memberikan dukungan teknis (IT Support) kepada...,5,1,...,5,Dki Jakarta,16.67,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,Yes,No
20249,a240c86a-eb17-4011-8d58-50b48b5f100c,2026-07-16T12:59:30+07:00,PENGELOLA KEHUMASAN,BALAI PEMASYARAKATAN KELAS II TERNATE,Kota Ternate,Bachelor,Komunikasi,1. Menyusun materi layanan informasi untuk med...,5,1,...,11,Maluku Utara,8.33,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,No,No
15585,a241492e-78dc-4fda-a1ca-11d0c7dd57d9,2026-07-16T12:05:25+07:00,Pengelola Kehumasan,RUMAH TAHANAN NEGARA KELAS IIB KABANJAHE,Kab. Karo,Bachelor,Komunikasi,"""1. Menyusun materi layanan informasi untuk me...",6,1,...,8,Sumatera Utara,11.11,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No


## 3.4 Feature Extraction
Extracting a new binary flag, `allows_all_majors`, by parsing the `job_description` column.

In [104]:
all_majors_condition = internship_positions.job_description.str.contains(
    r"semua\sjurusan|jurusan\sapa.*|all\smajors|any\smajor",
    case=False
)

internship_positions["allows_all_majors"] = np.where(all_majors_condition, "Yes", "No")

display(internship_positions.sample(5))

,job_id,published_at,job_title,company,regency_city,education_level,allowed_major,job_description,weekly_working_day,requested_quota,...,province,acceptance_percentage,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,allows_bachelor_level,allows_diploma_level,allows_profession_level,allows_all_majors
247,a23f5a97-1713-4bf2-a985-13e29253e25e,2026-07-16T11:58:22+07:00,PSIKIATER,LEMBAGA PEMASYARAKATAN TERBUKA KELAS IIB NUSAK...,Kab. Cilacap,Bachelor,Kedokteran,1. Menangani gangguan kesehatan jiwa warga bin...,5,1,...,Jawa Tengah,50.00,1 to 2,1 to 2,0 to 5,26 - 50%,Yes,No,No,No
12768,a2410a90-0f68-4ea0-9f76-15a7ff9d1852,2026-07-16T12:35:57+07:00,PENGELOLA KEUANGAN DAN ANGGARAN,KANTOR IMIGRASI KELAS II NON TPI BIMA,Kota Bima,Bachelor,Akuntansi,1. Menyusun rencana kebutuhan anggaran tahunan...,5,1,...,Nusa Tenggara Barat,12.50,1 to 2,1 to 2,6 to 10,11 - 25%,Yes,No,No,No
6459,a243e5bb-686b-4fc1-9acf-5c39bb5a6dc6,2026-07-16T12:02:12+07:00,DRID – Asisten Pengelolaan Data Kebijakan Daer...,Deputi Bidang Riset dan Inovasi Daerah,Kota Adm. Jakarta Pusat,"Bachelor, Diploma","Ekonomi Pembangunan, Sains Data, Administrasi ...",Membantu analisis data hasil riset dan inovasi...,5,1,...,Dki Jakarta,16.67,1 to 2,1 to 2,0 to 5,11 - 25%,Yes,Yes,No,No
19579,a241aa07-aac9-4a6c-849c-5a965a9c3507,2026-07-16T10:08:50+07:00,Seksi Pengiriman,Perum Percetakan Uang Ri,Kab. Karawang,"Diploma, Bachelor","Teknik Informatika, Informatika, Sistem Informasi",• Melakukan inventarisasi dan merapikan seluru...,5,1,...,Jawa Barat,9.09,1 to 2,1 to 2,6 to 10,0 - 10%,Yes,Yes,No,No
26279,a2414c35-be88-49bf-9c27-8665af5468c2,2026-07-16T13:12:33+07:00,Frontliner (FL),BPJS Kesehatan Kantor Cabang Curup,Kab. Rejang Lebong,"Diploma, Bachelor","Manajemen, Ilmu Komunikasi, Administrasi Bisni...",Membantu pelaksanaan kegiatan administratif da...,5,1,...,Bengkulu,4.76,1 to 2,1 to 2,11 to 20,0 - 10%,Yes,Yes,No,No


# 4. Schema Finalization
Reorganizing the final 14 columns into a logical analytical structure before exporting.

In [105]:
# Get all the columns
internship_positions.columns

Index(['job_id', 'published_at', 'job_title', 'company', 'regency_city',
       'education_level', 'allowed_major', 'job_description',
       'weekly_working_day', 'requested_quota', 'approved_quota',
       'applicant_count', 'province', 'acceptance_percentage',
       'requested_quota_category', 'approved_quota_category',
       'applicant_count_category', 'acceptance_percentage_category',
       'allows_bachelor_level', 'allows_diploma_level',
       'allows_profession_level', 'allows_all_majors'],
      dtype='str')

In [106]:
# Reorganize the position of the columns
final_cols = [
    "job_id",
    "published_at",
    "job_title",
    "company",
    "regency_city",
    "province",
    "allowed_major",
    "allows_all_majors",
    "allows_bachelor_level",
    "allows_diploma_level",
    "allows_profession_level",
    "job_description",
    "weekly_working_day",
    "requested_quota_category",
    "approved_quota_category",
    "applicant_count_category",
    "acceptance_percentage_category",
    "requested_quota",
    "approved_quota",
    "applicant_count",
    "acceptance_percentage",
]

internship_postings = internship_positions[final_cols]

display(internship_postings.sample(10))

,job_id,published_at,job_title,company,regency_city,province,allowed_major,allows_all_majors,allows_bachelor_level,allows_diploma_level,...,job_description,weekly_working_day,requested_quota_category,approved_quota_category,applicant_count_category,acceptance_percentage_category,requested_quota,approved_quota,applicant_count,acceptance_percentage
16271,a243ad8d-7c48-4abe-9c2f-0eb568f11dff,2026-07-16T09:58:13+07:00,Admin Finance,PT Gunanusa Eramandiri,Kab. Bekasi,Jawa Barat,Akuntansi,No,Yes,No,...,Staff Finance bertanggung jawab mengelola admi...,5,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,8,11.11
2852,a23f29c9-71c2-4fb5-9527-5ed1db572649,2026-07-16T12:36:02+07:00,Asisten Statistisi,BPS Kabupaten Maluku Tengah,Kab. Maluku Tengah,Maluku,"Sains Data, Ilmu Komputer, Teknik lndustri, Ek...",No,Yes,No,...,"Membantu pengumpulan, pengolahan, verifikasi, ...",5,3 to 10,3 to 10,11 to 20,26 - 50%,4,4,13,28.57
13735,a23f5394-7929-4767-be37-97d5fa0c44d1,2026-07-16T10:24:22+07:00,Strategic Busienss 3 - Admin Intern,Asuransi Tugu Pratama Indonesia,Kota Adm. Jakarta Selatan,Dki Jakarta,"Ekonomi, Akuntansi",No,Yes,No,...,Membantu pelaksanaan administrasi di Strategic...,5,1 to 2,1 to 2,6 to 10,11 - 25%,1,1,7,12.50
23245,a23f86a3-8d34-4df4-8dc1-8ea4ed617782,2026-07-16T16:23:31+07:00,Staf Dept. Pengembangan Aset dan Komersial,Pembangunan Perumahan Nasional,Kota Adm. Jakarta Timur,Dki Jakarta,"Arsitektur, Perencanaan Wilayah dan Kota",No,Yes,No,...,"""1. Membuat laporan & materi presentasi kajian...",5,1 to 2,1 to 2,11 to 20,0 - 10%,1,1,14,6.67
22990,a2395d07-3450-4fce-b93a-3205e643ccff,2026-07-23T20:26:35+07:00,Sales and Operations Support Division Intern -...,PT Superintending Company Of Indonesia,Kota Balikpapan,Kalimantan Timur,"Teknik Industri, Manajemen Bisnis",No,Yes,Yes,...,Mendukung kegiatan Sales and Operations Suppor...,5,1 to 2,1 to 2,21 to 50,0 - 10%,2,2,26,7.41
18014,a23f899a-e7b5-4ebe-9d21-f5e6d3b4774a,2026-07-16T10:08:18+07:00,Procurement Intern,PT. Tera Logistic Indonesia,Kota Adm. Jakarta Pusat,Dki Jakarta,"Teknik Mesin, Manajemen, Teknik Industri, Tekn...",No,Yes,No,...,Membantu kegiatan administrasi pengadaan baran...,5,1 to 2,1 to 2,6 to 10,0 - 10%,1,1,9,10.00
19928,a2414e76-9cff-4047-a464-de52fd5b4b16,2026-07-16T10:19:18+07:00,Staf Keuangan dan Akuntansi,PT. Abna Samanhudisautika Husada,Kab. Malang,Jawa Timur,"Manajemen Keuangan, Perpajakan, Akuntansi",No,Yes,No,...,Staf Keuangan dan Akuntansi bertugas mengelola...,5,3 to 10,3 to 10,21 to 50,0 - 10%,3,3,31,9.38
28145,a24357a4-8281-4f37-9e46-7664989aaf00,2026-07-16T12:09:43+07:00,Relationship Officer (RO),BPJS Kesehatan Kantor Cabang Surabaya,Kota Surabaya,Jawa Timur,"Manajemen, Administrasi Publik, Ilmu Komunikas...",No,Yes,Yes,...,Membantu pelaksanaan kegiatan administratif da...,5,1 to 2,1 to 2,21 to 50,0 - 10%,1,1,40,2.44
17026,a23f1412-5898-4b50-b05e-d6b505204bb2,2026-07-25T09:56:20+07:00,K3 OFFICER,PT Panca Budi Idaman Tbk,Kab. Pemalang,Jawa Tengah,"Kesehatan dan Keselamatan Kerja, Keselamatan K...",No,Yes,Yes,...,"Bertanggung jawab merencanakan, melaksanakan, ...",5,1 to 2,1 to 2,6 to 10,0 - 10%,1,1,9,10.00
3256,a23ee19f-99ba-49a4-8b34-7423707f9a21,2026-07-16T12:37:02+07:00,Asisten Statistisi,BPS Kota Bitung,Kota Bitung,Sulawesi Utara,"Informatika, Ekonomi Pembangunan, Sains Data, ...",No,Yes,No,...,1. Memahami Sistem Statistik Nasional\n2. Peng...,5,3 to 10,3 to 10,11 to 20,11 - 25%,3,3,11,25.00


In [107]:
# Load the final, clean data to a local directory
internship_postings.to_parquet(
    INTERIM_DATA_DIR / "internship_postings.parquet", index=False
) 